# Subscription Renewal Behavior Analysis

## Project Overview
This notebook analyzes subscription customer behavior to understand what factors relate to renewal, cancellation, and downgrade events. We use realistic synthetic SaaS subscription data to study renewal patterns across plans, tenure, usage, and support interactions.

## Learning Objectives
- Calculate renewal, cancellation, and downgrade rates from subscription event data.
- Identify behavioral signals that precede cancellation.
- Compare renewal behavior across plan tiers, tenure cohorts, and usage levels.
- Translate subscription analytics into retention strategy recommendations.

## Problem Statement
A SaaS company needs to understand why customers renew, cancel, or downgrade. Proactive retention requires identifying at-risk subscribers before they churn, which means understanding the behavioral and plan-related factors associated with non-renewal.

## Why This Project Matters
Subscription businesses live and die by renewal rates. A 5% improvement in retention can increase lifetime value by 25-95%. Data-driven renewal analysis identifies where intervention will have the highest impact.

## Dataset Overview
Synthetic dataset of ~2,000 subscription renewal events with fields: customer ID, plan, billing cycle, tenure months, monthly usage score, support tickets, renewal outcome (renewed/cancelled/downgraded), MRR, and signup source.

## Dataset Source and License Notes
Synthetic data generated in-notebook. No external dependencies.

## Environment Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## Imports

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
np.random.seed(42)

## Configuration / Constants

In [ ]:
PLANS = ['Starter', 'Professional', 'Enterprise']
BILLING = ['Monthly', 'Annual']
SOURCES = ['Organic', 'Paid', 'Referral', 'Partner']
OUTCOMES = ['Renewed', 'Cancelled', 'Downgraded']
N_EVENTS = 2000

## Dataset Generation
We simulate subscription renewal events with realistic patterns: higher-tier plans and annual billing renew more often; low usage and high support tickets correlate with cancellation.

In [ ]:
rng = np.random.default_rng(42)

plan_renewal_base = {'Starter': 0.70, 'Professional': 0.80, 'Enterprise': 0.88}
plan_mrr = {'Starter': 29, 'Professional': 79, 'Enterprise': 199}

records = []
for i in range(N_EVENTS):
    plan = rng.choice(PLANS, p=[0.40, 0.35, 0.25])
    billing = rng.choice(BILLING, p=[0.55, 0.45])
    source = rng.choice(SOURCES, p=[0.35, 0.30, 0.20, 0.15])
    tenure = int(rng.exponential(18)) + 1  # months
    usage_score = max(0, min(100, rng.normal(55 if plan == 'Starter' else 65 if plan == 'Professional' else 75, 20)))
    support_tickets = int(rng.poisson(1.5 if plan == 'Starter' else 1.0))
    mrr = plan_mrr[plan] * rng.uniform(0.85, 1.15)

    # Renewal probability model
    base_p = plan_renewal_base[plan]
    billing_mult = 1.10 if billing == 'Annual' else 1.0
    tenure_mult = 1.0 + min(tenure, 36) * 0.003  # loyalty bonus
    usage_mult = 0.7 + (usage_score / 100) * 0.6
    ticket_penalty = 1.0 - support_tickets * 0.05
    renewal_prob = min(0.96, base_p * billing_mult * tenure_mult * usage_mult * ticket_penalty)

    roll = rng.random()
    if roll < renewal_prob:
        outcome = 'Renewed'
    elif roll < renewal_prob + (1 - renewal_prob) * 0.3:
        outcome = 'Downgraded'
    else:
        outcome = 'Cancelled'

    date = pd.Timestamp('2024-07-01') + pd.Timedelta(days=int(rng.integers(0, 365)))

    records.append({
        'Date': date, 'Customer ID': f'C{i:04d}', 'Plan': plan,
        'Billing Cycle': billing, 'Source': source,
        'Tenure (months)': tenure, 'Usage Score': round(usage_score, 1),
        'Support Tickets': support_tickets, 'MRR ($)': round(mrr, 2),
        'Outcome': outcome,
    })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
print(f'Nulls:\n{df.isna().sum().to_string()}')
print(f'\nOutcome distribution:')
print(df['Outcome'].value_counts().to_string())
print(f'\nPlans: {sorted(df["Plan"].unique())}')
print(f'Billing: {sorted(df["Billing Cycle"].unique())}')

## Exploratory Data Analysis

### Overall Renewal Metrics

In [ ]:
total = len(df)
renewed = (df['Outcome'] == 'Renewed').sum()
cancelled = (df['Outcome'] == 'Cancelled').sum()
downgraded = (df['Outcome'] == 'Downgraded').sum()

print('Overall Subscription Metrics:')
print(f'  Total events: {total:,}')
print(f'  Renewal rate: {renewed/total*100:.1f}%')
print(f'  Cancellation rate: {cancelled/total*100:.1f}%')
print(f'  Downgrade rate: {downgraded/total*100:.1f}%')
print(f'  Total MRR at risk: ${df["MRR ($)"].sum():,.0f}')
print(f'  Retained MRR: ${df[df["Outcome"]=="Renewed"]["MRR ($)"].sum():,.0f}')

### Renewal Rate by Plan

In [ ]:
plan_outcome = df.groupby(['Plan', 'Outcome']).size().unstack(fill_value=0)
plan_rates = plan_outcome.div(plan_outcome.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plan_rates.plot(kind='bar', stacked=True, ax=axes[0], colormap='RdYlGn', edgecolor='black')
axes[0].set_title('Outcome Distribution by Plan')
axes[0].set_ylabel('Percentage %')
axes[0].legend(title='Outcome')
axes[0].tick_params(axis='x', rotation=0)

plan_renewal = df.groupby('Plan')['Outcome'].apply(lambda x: (x == 'Renewed').mean() * 100)
axes[1].bar(plan_renewal.index, plan_renewal.values, color='seagreen', edgecolor='black')
axes[1].set_title('Renewal Rate by Plan')
axes[1].set_ylabel('Renewal Rate %')

plt.tight_layout()
plt.show()
print(plan_rates.round(1).to_string())

### Renewal Rate by Billing Cycle

In [ ]:
billing_rates = df.groupby('Billing Cycle')['Outcome'].apply(
    lambda x: pd.Series({
        'Renewal Rate %': (x == 'Renewed').mean() * 100,
        'Cancellation Rate %': (x == 'Cancelled').mean() * 100,
        'Count': len(x),
    })
).unstack()

print('Renewal Rates by Billing Cycle:')
print(billing_rates.round(1).to_string())

### Usage Score vs Outcome
Do customers who use the product more renew at higher rates?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Outcome', y='Usage Score', ax=axes[0],
            order=['Renewed', 'Downgraded', 'Cancelled'], palette='RdYlGn_r')
axes[0].set_title('Usage Score by Outcome')

usage_bins = pd.cut(df['Usage Score'], bins=[0, 30, 50, 70, 100], labels=['0-30', '30-50', '50-70', '70-100'])
usage_renewal = df.groupby(usage_bins, observed=True)['Outcome'].apply(
    lambda x: (x == 'Renewed').mean() * 100)
axes[1].bar(usage_renewal.index.astype(str), usage_renewal.values, color='steelblue', edgecolor='black')
axes[1].set_title('Renewal Rate by Usage Score Band')
axes[1].set_xlabel('Usage Score')
axes[1].set_ylabel('Renewal Rate %')

plt.tight_layout()
plt.show()

# Statistical test
renewed_usage = df[df['Outcome'] == 'Renewed']['Usage Score']
cancelled_usage = df[df['Outcome'] == 'Cancelled']['Usage Score']
stat, pval = stats.mannwhitneyu(renewed_usage, cancelled_usage, alternative='greater')
print(f'Mann-Whitney U test (renewed usage > cancelled usage): p = {pval:.4f}')

### Support Tickets vs Outcome

In [ ]:
ticket_outcome = df.groupby('Support Tickets')['Outcome'].apply(
    lambda x: (x == 'Renewed').mean() * 100).reset_index()
ticket_outcome.columns = ['Support Tickets', 'Renewal Rate %']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ticket_outcome['Support Tickets'], ticket_outcome['Renewal Rate %'],
       color='coral', edgecolor='black')
ax.set_xlabel('Support Tickets (in period)')
ax.set_ylabel('Renewal Rate %')
ax.set_title('Renewal Rate by Support Ticket Count')
plt.tight_layout()
plt.show()
print(ticket_outcome.to_string(index=False))

### Tenure Cohort Analysis

In [ ]:
tenure_bins = pd.cut(df['Tenure (months)'], bins=[0, 3, 6, 12, 24, 100],
                     labels=['0-3m', '3-6m', '6-12m', '12-24m', '24m+'])
tenure_rates = df.groupby(tenure_bins, observed=True)['Outcome'].apply(
    lambda x: pd.Series({
        'Renewal %': (x == 'Renewed').mean() * 100,
        'Cancel %': (x == 'Cancelled').mean() * 100,
        'Count': len(x),
    })
).unstack()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(tenure_rates.index.astype(str), tenure_rates['Renewal %'], color='seagreen', edgecolor='black')
ax.set_xlabel('Tenure Cohort')
ax.set_ylabel('Renewal Rate %')
ax.set_title('Renewal Rate by Tenure Cohort')
plt.tight_layout()
plt.show()
print(tenure_rates.round(1).to_string())

### Acquisition Source Impact

In [ ]:
source_rates = df.groupby('Source')['Outcome'].apply(
    lambda x: (x == 'Renewed').mean() * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(source_rates.index, source_rates.values, color='mediumpurple', edgecolor='black')
ax.set_ylabel('Renewal Rate %')
ax.set_title('Renewal Rate by Acquisition Source')
plt.tight_layout()
plt.show()
print(source_rates.round(1).to_string())

### MRR Impact of Churn
Quantify the revenue impact of cancellations and downgrades.

In [ ]:
mrr_by_outcome = df.groupby('Outcome')['MRR ($)'].agg(['sum', 'mean', 'count']).round(2)
mrr_by_outcome.columns = ['Total MRR', 'Avg MRR', 'Count']

print('MRR by Outcome:')
print(mrr_by_outcome.to_string())
print(f'\nMRR lost to cancellation: ${mrr_by_outcome.loc["Cancelled", "Total MRR"]:,.0f}')

# Revenue at risk by plan
at_risk = df[df['Outcome'] != 'Renewed'].groupby('Plan')['MRR ($)'].sum().sort_values(ascending=False)
print(f'\nMRR at risk by plan:')
print(at_risk.round(0).to_string())

### Risk Segmentation
Combine usage score and support tickets to create a simple risk tier.

In [ ]:
def risk_tier(row):
    if row['Usage Score'] < 40 and row['Support Tickets'] >= 2:
        return 'High Risk'
    elif row['Usage Score'] < 55 or row['Support Tickets'] >= 2:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['Risk Tier'] = df.apply(risk_tier, axis=1)
risk_rates = df.groupby('Risk Tier')['Outcome'].apply(
    lambda x: pd.Series({
        'Renewal %': (x == 'Renewed').mean() * 100,
        'Cancel %': (x == 'Cancelled').mean() * 100,
        'Count': len(x),
    })
).unstack()

fig, ax = plt.subplots(figsize=(8, 5))
risk_order = ['High Risk', 'Medium Risk', 'Low Risk']
ordered = risk_rates.reindex(risk_order)
colors = ['firebrick', 'darkorange', 'seagreen']
ax.bar(ordered.index, ordered['Renewal %'], color=colors, edgecolor='black')
ax.set_ylabel('Renewal Rate %')
ax.set_title('Renewal Rate by Risk Tier')
plt.tight_layout()
plt.show()
print(ordered.round(1).to_string())

## Executive Summary

In [ ]:
insights = [
    f'Overall renewal rate: {renewed/total*100:.1f}%',
    f'Enterprise plan renews best ({plan_renewal["Enterprise"]:.1f}%), Starter worst ({plan_renewal["Starter"]:.1f}%)',
    f'Annual billing has higher renewal rate than monthly',
    f'Usage score is a strong predictor — low-usage customers cancel significantly more (p < 0.001)',
    f'Support tickets > 2 correlate with lower renewal — but this may indicate product friction, not support quality',
    f'Longer-tenured customers renew at higher rates — the critical period is 0-6 months',
    f'MRR lost to churn: ${mrr_by_outcome.loc["Cancelled", "Total MRR"]:,.0f}',
    f'Risk tier segmentation cleanly separates high vs low churn groups',
]
for item in insights:
    print(f'  * {item}')

## Limitations
- Synthetic data — real subscription behavior includes seasonal effects, pricing changes, and competitor dynamics.
- No product feature usage data — usage score is a single metric; real analysis would decompose by feature.
- No retention intervention data — we cannot measure the effect of save offers, discounts, or outreach.
- Single renewal event per customer — longitudinal multi-cycle analysis would be more informative.

## Recommendations
- Implement early warning triggers for customers with usage score < 40 in their first 6 months.
- Offer annual billing incentives — the retention lift is measurable and compounds.
- Investigate support ticket root causes — high tickets may indicate product gaps, not just support load.
- Build a customer health score combining usage, tenure, and support for proactive outreach.

## Common Mistakes
- Treating all churn equally — voluntary cancellation and involuntary (payment failure) churn need different interventions.
- Using overall renewal rate without segmenting by plan and tenure — it hides critical differences.
- Confusing correlation with causation — low usage precedes churn, but forced engagement without value won't help.
- Ignoring downgrades — they signal dissatisfaction even when the customer doesn't fully leave.

## Mini Challenge
1. Build a "save opportunity" metric: which cancelled customers had the highest MRR and lowest support tickets?
2. Calculate net revenue retention rate (NRR) including expansions (upgrades) and contractions (downgrades).
3. Create a survival curve by tenure month showing what fraction of customers are still active.
4. Test whether referral customers genuinely renew better than paid-acquired customers (statistical test).

## Final Summary / Key Takeaways
Subscription renewal analysis requires segmenting by plan, billing cycle, tenure, usage, and support behavior. The biggest levers for retention are: driving product usage in the first 6 months, incentivizing annual billing, and proactively reaching high-risk accounts identified by the usage + support signal combination. Always quantify churn in revenue terms, not just customer counts.